# 14 · Evaluation of the final runs on the jittered crops (no training)
**Runtime: T4 GPU** (CPU works but is slow). Run all **after** notebooks 13_train_session_A/B have finished (it can also be run
earlier: it only evaluates runs that are finished, and a later Run all adds the rest). Read-only for all run folders.
Writes `results/p2_jit_eval/`.

For every finished run in `results/p2_jit_runs_train` and `results/p2_jit_runs_val`, on the CBIS-DDSM test fold of that run and on
all DMID crops (external test, no fine-tuning; `data/external/dmid_mass_256_jit/roi`):
1. per-image Dice, IoU and HD95 of the normal prediction; the CBIS mean is compared with the value stored at training time (reload check);
2. severing: constant code (mean training code) and swap (code of a random training image) with **10 different random draws**
   (draw 0 = the order used at training time), Dice per image and draw;
3. prediction-map change for const and swap draw 0: fraction of pixels whose binary prediction flips, and mean |Δp|.

Resumable: each run writes its own file; finished files are skipped.


In [ ]:
# 1 · Drive, paths, imports
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time, re
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
DMID_DIR = f'{PROJECT}/data/external/dmid_mass_256_jit/roi'
SETS     = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val'}
DEDUP    = f'{PROJECT}/results/p2_analysis_v2/dmid_dedup.csv'      # same DMID mass ids as before (checked in notebook 11)
OUT      = f'{PROJECT}/results/p2_jit_eval'
CODE_DIR = f'{PROJECT}/code'
QSEG_DIR = '/content/drive/MyDrive/quantum-placement/code'
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True)
    assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (QSEG_DIR, CODE_DIR):
    if p not in sys.path: sys.path.insert(0, p)
for s in SETS: os.makedirs(f'{OUT}/{s}', exist_ok=True)
import importlib, tensorflow as tf, qseg2; importlib.reload(qseg2)
from qseg.config import Config
cfg = Config(work_dir=OUT)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# 2 · Data
meta = pd.read_csv(f'{DMID_DIR}/metadata.csv')
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
XD = np.stack([_png(f'{DMID_DIR}/images/{r}.png', False) for r in meta.roi_id])
YD = np.stack([_png(f'{DMID_DIR}/masks/{r}.png', True) for r in meta.roi_id])
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
dd = pd.read_csv(DEDUP); keep = dict(zip(dd.roi_id, dd.keep))
assert set(meta.roi_id) == set(keep), 'DMID ids differ from the dedup table'
print('DMID', XD.shape, '| kept after dedup:', int(sum(keep[r] for r in meta.roi_id)), '| CBIS', XC.shape)


In [ ]:
# 3 · Evaluate every finished run (resumable)
K_DRAWS = 10
def check_run(set_name, run_dir):
    run = os.path.basename(run_dir); dst = f'{OUT}/{set_name}/{run}.csv'
    if os.path.exists(dst): return None
    s = json.load(open(f'{run_dir}/summary.json'))
    name, fold, seed = s['model'], int(s['fold']), int(s['seed'])
    tf.keras.backend.clear_session()
    m = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m.load_weights(f'{run_dir}/best.weights.h5')
    tr, va, te = qseg2.fold_split(DFC, fold)
    has_sever = 'sever' in [l.name for l in m.layers]
    if has_sever:
        qseg2.set_sever_bank(m, XC[tr], cfg.batch_size, seed)                 # same bank recipe as training
        sev = m.get_layer('sever'); bank0 = sev.bank.numpy().copy()
    out = []
    for ds, X, Y, ids in (('CBIS', XC[te], YC[te], DFC.roi_id.values[te]), ('DMID', XD, YD, meta.roi_id.values)):
        P0 = m.predict(X, batch_size=cfg.batch_size, verbose=0)
        t = qseg2.per_image_scores(Y, P0, cfg.threshold)
        t.insert(0, 'roi_id', ids)
        if has_sever:
            sev.bank.assign(bank0)
            Pc = qseg2.predict_with_mode(m, X, 1, cfg.batch_size)
            t['dice_const'] = qseg2.per_image_scores(Y, Pc, cfg.threshold).dice.values
            t['flip_const'] = ((Pc[..., 0] >= cfg.threshold) != (P0[..., 0] >= cfg.threshold)).mean((1, 2))
            t['absdp_const'] = np.abs(Pc - P0)[..., 0].mean((1, 2))
            for d in range(K_DRAWS):
                perm = np.arange(len(bank0)) if d == 0 else np.random.default_rng(1000 + d).permutation(len(bank0))
                sev.bank.assign(bank0[perm])
                Ps = qseg2.predict_with_mode(m, X, 2, cfg.batch_size)
                t[f'dice_swap{d}'] = qseg2.per_image_scores(Y, Ps, cfg.threshold).dice.values
                if d == 0:
                    t['flip_swap'] = ((Ps[..., 0] >= cfg.threshold) != (P0[..., 0] >= cfg.threshold)).mean((1, 2))
                    t['absdp_swap'] = np.abs(Ps - P0)[..., 0].mean((1, 2))
            sev.bank.assign(bank0)
        t.insert(0, 'dataset', ds)
        t['keep'] = True if ds == 'CBIS' else t.roi_id.map(keep).astype(bool)
        out.append(t)
    out = pd.concat(out, ignore_index=True)
    out['model'], out['fold'], out['seed'], out['set'] = name, fold, seed, set_name
    out['cbis_dice_training'] = float(s['dice_per_image'])
    out['cbis_dice_reloaded'] = float(out.loc[out.dataset == 'CBIS', 'dice'].mean())
    out['epochs_run'], out['best_epoch'] = s.get('epochs_run'), s.get('best_epoch')
    out.to_csv(dst, index=False)
    return out
n_new = 0
for set_name, root in SETS.items():
    runs = [d for d in sorted(glob.glob(f'{root}/*_fold*_seed*'))
            if os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5')]
    print(set_name, len(runs), 'finished runs')
    for d in runs:
        t0 = time.time(); r = check_run(set_name, d)
        if r is not None:
            n_new += 1
            dm = r[(r.dataset == 'DMID') & r.keep]
            print(f"{set_name:6s} {os.path.basename(d):22s} reload diff {r.cbis_dice_reloaded.iloc[0] - r.cbis_dice_training.iloc[0]:+.5f} "
                  f"| CBIS {r.cbis_dice_reloaded.iloc[0]:.4f} | DMID {dm.dice.mean():.4f} ({time.time() - t0:.0f} s)", flush=True)
print('evaluation finished;', n_new, 'new runs')


In [ ]:
# 4 · Consolidate + summary tables
allc = pd.concat([pd.read_csv(f) for s in SETS for f in sorted(glob.glob(f'{OUT}/{s}/*.csv'))], ignore_index=True)
allc.to_csv(f'{OUT}/eval_per_image_all.csv', index=False)
rel = allc.groupby(['set', 'model', 'fold', 'seed'])[['cbis_dice_training', 'cbis_dice_reloaded']].first().reset_index()
rel['diff'] = rel.cbis_dice_reloaded - rel.cbis_dice_training
rel.to_csv(f'{OUT}/reload_check.csv', index=False)
print('runs evaluated:', rel.groupby('set').size().to_dict())
print('reload |diff|: max', round(float(rel['diff'].abs().max()), 6), '| median', round(float(rel['diff'].abs().median()), 6))
k = allc[allc.keep == True]
cols = [c for c in ['dice', 'iou', 'dice_const', 'dice_swap0'] if c in k]
run_means = k.groupby(['set', 'dataset', 'model', 'fold', 'seed'])[cols].mean()
tab = run_means.groupby(['set', 'dataset', 'model']).mean().round(4)
tab['n_runs'] = run_means.groupby(['set', 'dataset', 'model']).size()
pd.set_option('display.width', 200)
print(tab.to_string())
tab.to_csv(f'{OUT}/summary_by_model.csv')
print('Files to share:', sorted(os.listdir(OUT)))


In [ ]:
# 5 · Auto-shutdown: save Drive, then release the GPU
import time
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()
